# Section 3: workload characteristics

Container counts, resource allocations, termination types and creation times (Tables 2, 3 and 5; Figures 2-6). Run by `run_manuscript.py`; outputs in `05_Results/section3/execution_runs/01_usage`.

## Setup

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
# Execution scope and output capture (no change to policy calculations).
import json as _mf_json
import re as _mf_re
import numpy as _mf_np
import pandas as _mf_pd
import matplotlib.pyplot as _mf_plt
from matplotlib.figure import Figure as _mf_Figure
_MF_OUTPUT = ROOT / '05_Results/section3/execution_runs/01_usage'
_MF_OUTPUT.mkdir(parents=True,exist_ok=True)
_MF_CAPTURE = _MF_OUTPUT / 'capture'
_MF_CAPTURE.mkdir(exist_ok=True)
_MF_MODE = 'corrected'
_MF_CELL = 0
_MF_ERRORS = []
PROCESSED_DATA = ROOT / '03_ProcessedData'
RESULTS_SECTION3 = _MF_OUTPUT
RESULTS_SECTION4 = _MF_OUTPUT
RESULTS_REVISION = _MF_OUTPUT
_MF_REAL_READ_CSV = _mf_pd.read_csv
def _mf_read_csv(*args,**kwargs):
    try:
        value = _MF_REAL_READ_CSV(*args,**kwargs)
    except UnicodeDecodeError:
        # Encoding fallback for cp949 CSV files (I/O only).
        value = _MF_REAL_READ_CSV(*args,**{**kwargs,"encoding":"cp949"})
    for _stamp in ['created_time','terminated_time']:
        if _stamp in value.columns:
            _dt = _mf_pd.to_datetime(value[_stamp],errors='raise')
            for _unit in ['year','month','day','hour']:
                _name=f'{_stamp}({_unit})'
                if _name not in value.columns: value[_name]=getattr(_dt.dt,_unit)
    return value
_mf_pd.read_csv = _mf_read_csv
_mf_orig_savefig = _mf_Figure.savefig
def _mf_savefig(self,fname,*args,**kwargs):
    if isinstance(fname,(str,Path)):
        target=Path(fname)
        if not target.is_absolute() or not target.resolve().is_relative_to(_MF_OUTPUT):
            target = _MF_OUTPUT / target.name
        target.parent.mkdir(parents=True,exist_ok=True)
        fname=target
    return _mf_orig_savefig(self,fname,*args,**kwargs)
_mf_Figure.savefig = _mf_savefig
def _mf_capture_cell(cell_number):
    import warnings as _mf_warnings
    with _mf_warnings.catch_warnings():
        _mf_warnings.simplefilter('ignore')
        for name,value in list(globals().items()):
            if name.startswith('_'):continue
            if isinstance(value,_mf_pd.DataFrame) and len(value) <= 1000 and len(value.columns) <= 80:
                value.to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
            elif isinstance(value,_mf_pd.Series) and len(value) <= 1000:
                value.to_frame().to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
        for no in _mf_plt.get_fignums():
            figure=_mf_plt.figure(no)
            if not figure.axes:continue
            path=_MF_CAPTURE/f'cell{cell_number:02d}_figure{no}.png'
            _mf_orig_savefig(figure,path,dpi=110,bbox_inches='tight')
            lines=[]
            for axno,axis in enumerate(figure.axes):
                for lineno,line in enumerate(axis.get_lines()):
                    x,y=line.get_data()
                    if len(x)!=len(y) or len(x)>100000:continue
                    if len(x):
                        lines.extend((axno,lineno,line.get_label(),i,a,b) for i,(a,b) in enumerate(zip(x,y)))
            if lines:
                _mf_pd.DataFrame(lines,columns=['axis','line','label','point','x','y']).to_csv(path.with_suffix('.csv'),index=False)


# Native Matplotlib export; no plot or calculation changes.
_MF_ORIG_SHOW=_mf_plt.show
_MF_FIG_COUNTER={}
def _mf_show(*args,**kwargs):
    for _number in _mf_plt.get_fignums():
        _figure=_mf_plt.figure(_number)
        if not _figure.axes: continue
        _MF_FIG_COUNTER[_MF_CELL]=_MF_FIG_COUNTER.get(_MF_CELL,0)+1
        _stem=f'cell{_MF_CELL:02d}_plot{_MF_FIG_COUNTER[_MF_CELL]:02d}'
        _folder=_MF_OUTPUT/'figures'; _folder.mkdir(exist_ok=True)
        _mf_orig_savefig(_figure,_folder/(_stem+'.png'),dpi=150,bbox_inches='tight')
        _mf_orig_savefig(_figure,_folder/(_stem+'.pdf'),bbox_inches='tight')
        if _MF_CELL in [14, 16, 30]:
            _artist_rows = []
            for _axis_index, _axis in enumerate(_figure.axes):
                for _line in _axis.get_lines():
                    _x, _y = _line.get_data()
                    for _point, (_a, _b) in enumerate(zip(_x, _y)):
                        _artist_rows.append((_axis_index, 'line', str(_line.get_label()), _point, _a, _b))
                for _container in _axis.containers:
                    if not hasattr(_container, 'patches'): continue
                    for _point, _patch in enumerate(_container.patches):
                        _artist_rows.append((_axis_index, 'bar', str(_container.get_label()), _point,
                                             _patch.get_x() + _patch.get_width()/2, _patch.get_height()))
            _mf_pd.DataFrame(_artist_rows, columns=['axis','kind','series','point','x','y']).to_csv(
                _folder/(_stem+'_artists.csv'), index=False)
    return _MF_ORIG_SHOW(*args,**kwargs)
_mf_plt.show=_mf_show


In [ ]:
_MF_CELL=2
# # 구글드라이브 연결
# from google.colab import drive



In [ ]:
_mf_capture_cell(2)


In [ ]:
_MF_CELL=3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'] # Matplotlib 기본 색상 가져오기
import matplotlib.ticker as mtick
from datetime import datetime
import time
import re
from functools import reduce

import warnings
warnings.filterwarnings('ignore')
import matplotlib.cbook

import os

plt.rcParams['font.family'] = 'sans-serif'  # 기본 sans-serif 폰트 사용
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans']  # 대체 폰트
plt.rcParams['axes.unicode_minus'] = False  # 음수 기호 문제 해결


In [ ]:
_mf_capture_cell(3)


In [ ]:
_MF_CELL=4
# ==============================================================================
# 0) 공통: 환경 판별 + BASE 설정 (코랩/로컬)
# ==============================================================================

from pathlib import Path
import os

def is_colab():
    return Path("/content").exists() and (
        "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
    )

COLAB_BASE = Path(str(PROCESSED_DATA))
LOCAL_BASE = Path(str(PROCESSED_DATA))

BASE = COLAB_BASE if is_colab() else LOCAL_BASE

# (선택) 코랩이면 드라이브 마운트
if is_colab():
    from google.colab import drive
    if not Path("/content/gdrive").exists():
        drive.mount("/content/gdrive")

print("✅ Running on:", "Colab" if is_colab() else "Local")
print("✅ BASE =", BASE)

# ==============================================================================
# 1) 라이브러리 (필요한 것만 우선)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 코랩에서만 매직 사용 (로컬 VSCode에선 에러 방지)
if is_colab():
    get_ipython().run_line_magic("matplotlib", "inline")

import warnings
warnings.filterwarnings("ignore")

# 폰트/마이너스 설정
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# ==============================================================================
# 2) 데이터 로드 (UsageLog)
# ==============================================================================

USAGE_LOG_PATH = BASE / "UsageLog20230401_20251130.csv"

df_selected = pd.read_csv(USAGE_LOG_PATH, sep=",")
print("✅ df_selected loaded:", df_selected.shape)
print("✅ columns:", list(df_selected.columns))


In [ ]:
_mf_capture_cell(4)


In [ ]:
_MF_CELL=5
df_selected.columns


In [ ]:
_mf_capture_cell(5)


In [ ]:
_MF_CELL=6
# 폰트 사이즈 조절
plt.rcParams['font.size'] = 15  # 모든 글자의 크기를 동일하게 설정
plt.rcParams['axes.titlesize'] = 15  # 그래프 제목
plt.rcParams['axes.labelsize'] = 15  # 축 레이블
plt.rcParams['xtick.labelsize'] = 15  # x축 눈금
plt.rcParams['ytick.labelsize'] = 15  # y축 눈금
plt.rcParams['lines.linewidth'] = 3   # 선 그래프 굵기 조절


In [ ]:
_mf_capture_cell(6)


In [ ]:
_MF_CELL=7
# GPU를 사용한 JOB과 CPU를 사용하는 Data를 별도로 저장한다.
df_selected_gpu = df_selected[df_selected['gpu_allocated(EA)']!=0]
df_selected_cpu = df_selected[df_selected['gpu_allocated(EA)']==0]


In [ ]:
_mf_capture_cell(7)


# TABLE 2
## CONTAINER COUNTS BY GPU USAGE.


In [ ]:
_MF_CELL=9
import numpy as np
import pandas as pd

col = "gpu_allocated(group)"
# Include non-integer allocations above one GPU as "1.x–3.x GPU".
valid_groups = ["0.0_GPU", "0.x_GPU", "1.0_GPU", "2.0_GPU", "3.0_GPU", "4.0_GPU", "1.x+_GPU"]

def _section3_analysis_population(frame):
    allocation = pd.to_numeric(frame["gpu_allocated(EA)"], errors="raise")
    assert np.isfinite(allocation).all() and allocation.ge(0).all()
    above_one_noninteger = allocation.gt(1) & allocation.mod(1).ne(0)
    selected = frame.copy()
    selected.loc[above_one_noninteger, col] = "1.x+_GPU"
    assert selected[col].isin(valid_groups).all(), "Unexpected allocation category"
    return selected

# Only Table 2 includes non-integer allocations above one GPU.
# Preserve the complete scheduler input; filter only the Section 3 analysis cohort.
df_clean = _section3_analysis_population(df_selected)
excluded_n = len(df_selected) - len(df_clean)
print(f"Table 2 population: {len(df_selected):,}; other Section 3 analyses: {len(df_clean):,}; excluded: {excluded_n:,}")

# Table 2: GPU allocation >0 versus no GPU, including 1.x/2.x/3.x.
df_full_counts = df_selected.copy()
allocation = pd.to_numeric(df_full_counts["gpu_allocated(EA)"], errors="raise")
df_full_counts["gpu_class"] = np.where(allocation.eq(0), "Non-GPU", "GPU-enabled")
gpu_class_table = (
    df_full_counts["gpu_class"].value_counts().rename_axis("gpu_class")
    .reset_index(name="count")
)
gpu_class_table["ratio(%)"] = (gpu_class_table["count"] / gpu_class_table["count"].sum() * 100).round(2)
gpu_class_table = gpu_class_table.set_index("gpu_class").loc[["Non-GPU", "GPU-enabled"]].reset_index()

# Allocation-size comparisons retain 0.x and integer GPU groups.
df_gpu = df_clean[df_clean[col].ne("0.0_GPU")].copy()
cnt = df_gpu[col].value_counts().sort_values(ascending=False)
gpu_group_table = pd.DataFrame({
    "container_count(EA)": cnt.astype(int),
    "container_ratio(%)": (cnt / cnt.sum() * 100).round(2),
})
gpu_group_table.loc["Total"] = [int(cnt.sum()), 100.00]
gpu_class_table, gpu_group_table


In [ ]:
_mf_capture_cell(9)


In [ ]:
_MF_CELL=10
# Non-integer allocations above one GPU are included here (Table 2 only).
# Table 2 is computed above using the numeric GPU allocation, not a nullable group.


In [ ]:
_mf_capture_cell(10)


# TABLE 3
## DISTRIBUTION OF CONTAINERS BY GPU ALLOCATION SIZE.


In [ ]:
_MF_CELL=12
# 0.0_GPU 제외
df_tmp = df_clean[df_clean["gpu_allocated(group)"] != "0.0_GPU"].copy()

# 집계 (count만 한 번 구해서 재사용)
cnt = df_tmp["gpu_allocated(group)"].value_counts().sort_values(ascending=False)

count_con_all = pd.DataFrame({
    "container_count(EA)": cnt.astype(int),
    "container_ratio(%)": (cnt / cnt.sum() * 100).round(2),
})

# 합계 행 추가
count_con_all.loc["Total"] = {
    "container_count(EA)": int(cnt.sum()),
    "container_ratio(%)": 100.00,
}

count_con_all


In [ ]:
_mf_capture_cell(12)


# FIGURE 4
## 0306_container_creation_by_time



In [ ]:
_MF_CELL=14
# 0306 시간대별 컨테이너 생성 수 그래프

# 데이터 준비
fig, ax = plt.subplots(figsize=(12, 6))
pd.DataFrame(df_clean[['created_time(hour)', 'device_type']]
             .value_counts(), columns=['count']) \
    .unstack('device_type') \
    .plot(ax=ax)

# ✅ 가로 레전드 + 위로 빼기
ax.legend(
    ['Non-GPU Containers', 'GPU-Enabled Containers'],
    loc='upper center',
    # bbox_to_anchor=(0.0, 1.18),   # 1.10~1.25 사이에서 취향대로 조절
    ncol=2,
    fontsize=23,
    frameon=True
)

# 축 레이블 설정
ax.set_ylabel('Number of Containers', fontsize=30)
ax.set_xlabel('Time of Day', fontsize=30)

# X축 눈금 설정 (2시간 간격)
xticks = range(0, 24, 2)
ax.set_xticks(xticks)
ax.set_xticklabels(xticks, fontsize=30)
ax.tick_params(axis='y', labelsize=30)

# 그래프 회전 및 범위 설정
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.ylim(0, 2000)

# 제목 추가 (선택적)
# ax.set_title('Container Creation by Hour', fontsize=30)

# 레이아웃 조정 및 그래프 출력
plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(14)


# FIGURE 3
## Exported as 0303_containers_vs_conference_schedule_year_month


In [ ]:
_MF_CELL=16
# 1. 첫 번째 그래프 데이터 준비 (연, 월별 컨테이너 수)
df_selected_group = pd.DataFrame(
    df_clean[['created_time(year)', 'created_time(month)', 'device_type']].value_counts(),
    columns=['count']
).unstack('device_type')

# 첫 번째 그래프의 인덱스를 연-월 형식으로 변환
df_selected_group.index = [f"{year}-{month:02d}" for year, month in df_selected_group.index]


# 3. 그래프 생성 및 결합
fig, ax1 = plt.subplots(figsize=(12, 6))

# 첫 번째 그래프: 라인 그래프 (컨테이너 수)
df_selected_group.plot(ax=ax1, legend=False, linewidth=3)

ax1.set_zorder(1)
ax1.patch.set_visible(False)
ax1.set_xlabel('Year-Month', fontsize=25)
ax1.set_ylabel('Number of Containers', fontsize=25)
ax1.tick_params(axis='x', labelsize=25)
ax1.tick_params(axis='y', labelsize=25)
# ✅ 가로 레전드 + 위로 빼기
ax1.legend(
    ['Non-GPU Containers', 'GPU-Enabled Containers'],
    loc='upper center',
    # bbox_to_anchor=(0.0, 1.18),   # 1.10~1.25 사이에서 취향대로 조절
    ncol=2,
    fontsize=23,
    frameon=True
)
ax1.set_ylim(0,1200)

# X축 레이블 2개월 단위로 설정
xticks = range(0, len(df_selected_group.index), 2)
ax1.set_xticks(xticks)
ax1.set_xticklabels([df_selected_group.index[i] for i in xticks], rotation=45, ha='center', fontsize=25)

# 레이아웃 정리 및 출력
plt.yticks(rotation=0)
fig.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(16)


# FIGURE 2
# 0301_gpu_count_usage_time_cdf


In [ ]:
_MF_CELL=18
# GPU 장수를 0.x장, 1장, 2장, 3장, 4장 으로 구간별 사용시간을 추출한다.
gpu_0 = df_clean[df_clean['gpu_allocated(group)']=='0.0_GPU']
gpu_0_x = df_clean[df_clean['gpu_allocated(group)']=='0.x_GPU']
gpu_1 = df_clean[df_clean['gpu_allocated(group)']=='1.0_GPU']
gpu_2 = df_clean[df_clean['gpu_allocated(group)']=='2.0_GPU']
gpu_3 = df_clean[df_clean['gpu_allocated(group)']=='3.0_GPU']
gpu_4 = df_clean[df_clean['gpu_allocated(group)']=='4.0_GPU']
gpu_1x = df_clean[df_clean['gpu_allocated(group)']=='1.x+_GPU']  # 


In [ ]:
_mf_capture_cell(18)


In [ ]:
_MF_CELL=19
# CDF 계산
x0 = np.sort(gpu_0['used_time(minute)'])
y0 = np.arange(1, len(x0)+1) / len(x0)

x1 = np.sort(gpu_0_x['used_time(minute)'])
y1 = np.arange(1, len(x1)+1) / len(x1)

x2 = np.sort(gpu_1['used_time(minute)'])
y2 = np.arange(1, len(x2)+1) / len(x2)

x3 = np.sort(gpu_2['used_time(minute)'])
y3 = np.arange(1, len(x3)+1) / len(x3)

x4 = np.sort(gpu_3['used_time(minute)'])
y4 = np.arange(1, len(x4)+1) / len(x4)

x5 = np.sort(gpu_4['used_time(minute)'])
y5 = np.arange(1, len(x5)+1) / len(x5)

x6 = np.sort(gpu_1x['used_time(minute)'])  # 1.x–3.x GPU
y6 = np.arange(1, len(x6)+1) / len(x6)

# 0 또는 음수 제거 (로그 스케일 불가)
x0 = x0[x0 > 0]
x1 = x1[x1 > 0]
x2 = x2[x2 > 0]
x3 = x3[x3 > 0]
x4 = x4[x4 > 0]
x5 = x5[x5 > 0]
x6 = x6[x6 > 0]

# Plot
# plt.figure(figsize=(12, 6))
plt.figure(figsize=(12, 8))

plt.plot(x0, y0, label='0 GPU', linewidth=3)
plt.plot(x1, y1, label='0.x GPU', linewidth=3)
plt.plot(x2, y2, label='1 GPU', linewidth=3)
plt.plot(x3, y3, label='2 GPU', linewidth=3)
plt.plot(x4, y4, label='3 GPU', linewidth=3)
plt.plot(x5, y5, label='4 GPU', linewidth=3)
plt.plot(x6, y6, label='1.x–3.x GPU', linewidth=3)

plt.xscale('log')  # 로그 스케일
plt.xlabel('Run Time (minutes)', fontsize=24)
plt.ylabel('Cumulative Fraction of Containers', fontsize=23)
plt.legend(fontsize=23)
plt.tick_params(axis='x', labelsize=25)
plt.tick_params(axis='y', labelsize=25)

plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(19)


# 0308 종료사유별 컨테이너 분포 그래프
## 0308_container_distribution_by_termination_reason


In [ ]:
_MF_CELL=21
# UsageLog 로드 (코랩/로컬 공통)
df_1 = _section3_analysis_population(pd.read_csv(BASE / "UsageLog20230401_20251130.csv"))


In [ ]:
_mf_capture_cell(21)


In [ ]:
_MF_CELL=22
# gpu_allocated(group) 값 변경
df_1['gpu_allocated(group)'] = df_1['gpu_allocated(group)'].replace({
    '0.0_GPU': '0 GPU',
    '0.x_GPU': '0.x GPU',
    '1.0_GPU': '1 GPU',
    '2.0_GPU': '2 GPU',
    '3.0_GPU': '3 GPU',
    '4.0_GPU': '4 GPU',
    '1.x+_GPU': '1.x–3.x GPU'
})


In [ ]:
_mf_capture_cell(22)


In [ ]:
_MF_CELL=23
df_1_gpu = df_1[df_1['gpu_allocated(group)'] != '0 GPU']
df_1_cpu = df_1[df_1['gpu_allocated(group)'] == '0 GPU']


In [ ]:
_mf_capture_cell(23)


In [ ]:
_MF_CELL=24
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 2. 집계 및 피벗 테이블 생성
df_analysis = df_1[df_1['status_info(group)'] != 'Error']
grouped = df_analysis.groupby(['gpu_allocated(group)', 'status_info(group)']).size().reset_index(name='count')
pivot_table = grouped.pivot(index='gpu_allocated(group)', columns='status_info(group)', values='count').fillna(0)

# 3. 데이터 정렬 및 비율 계산
order = ['0 GPU', '0.x GPU', '1 GPU', '2 GPU', '3 GPU', '4 GPU', '1.x–3.x GPU']  # 
actual_order = [o for o in order if o in pivot_table.index]
pivot_table = pivot_table.reindex(actual_order)

total_counts = pivot_table.sum(axis=1)

# ✅ Auto-Stop Ratio를 %로 변환
auto_ratio_pct = (pivot_table['Auto-stop'] / total_counts) * 100

display_labels = [x.replace('_GPU', '').replace('.0', '') for x in actual_order]

# 4. 시각화
fig, ax1 = plt.subplots(figsize=(14, 7))

pivot_table[['Auto-stop', 'Manual-stop']].plot(kind='bar', ax=ax1, width=0.8)

ax1.set_xlabel('GPU Count Per Container', fontsize=25)
ax1.set_ylabel('Number of Containers', fontsize=25)
ax1.set_xticklabels(display_labels, rotation=0, fontsize=25)
ax1.tick_params(axis='y', labelsize=25)
ax1.legend(['Auto-stop', 'Manual-stop'], loc='upper left', fontsize=20)
ax1.set_ylim(0, pivot_table.values.max() * 1.3)

# ✅ 우측 y축: 0~110% + 표시도 %
ax2 = ax1.twinx()
x = np.arange(len(actual_order))  # ✅ 실제 막대 개수와 동일하게
# Auto-stop 막대 색(첫 번째 시리즈의 첫 번째 bar 색) 추출
auto_color = ax1.containers[0].patches[0].get_facecolor()

auto_ratio_pct = (pivot_table['Auto-stop'] / total_counts) * 100

ax2.plot(
    x,
    auto_ratio_pct.values,
    color='black',            # ✅ 라인(점선) 검정
    linestyle='--',
    linewidth=2,
    marker='o',
    markerfacecolor=auto_color,  # ✅ 점 안쪽은 Auto-stop 색
    markeredgecolor='black',     # ✅ 점 테두리는 검정
    markeredgewidth=1.0,
    markersize=10,
    label='Auto-Stop Ratio (%)'
)
ax2.set_ylim(0, 20)
ax2.set_ylabel('Auto-Stop Ratio (%)', fontsize=25, labelpad=20)
ax2.tick_params(axis='y', labelsize=25)
ax2.set_yticks(np.arange(0, 20, 5))  # 0,20,40,60,80,100

ax2.legend(loc='upper right', fontsize=20)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(24)


# TABLE 5
## CONTAINER TERMINATIONS BY GPU COUNT PER CONTAINER AND TERMINATION TYPE.


In [ ]:
_MF_CELL=26

# 1) 그래프와 동일한 집계 로직
df_analysis = df_1[df_1['status_info(group)'] != 'Error'].copy()

grouped = (
    df_analysis
    .groupby(['gpu_allocated(group)', 'status_info(group)'])
    .size()
    .reset_index(name='count')
)

pivot_table = (
    grouped
    .pivot(index='gpu_allocated(group)', columns='status_info(group)', values='count')
    .fillna(0)
)

# 그래프와 동일한 순서
order = ['0 GPU', '0.x GPU', '1 GPU', '2 GPU', '3 GPU', '4 GPU', '1.x–3.x GPU']  # 
actual_order = [o for o in order if o in pivot_table.index]
pivot_table = pivot_table.reindex(actual_order).fillna(0)

# 혹시 컬럼이 없을 때 대비
if 'Auto-stop' not in pivot_table.columns:
    pivot_table['Auto-stop'] = 0
if 'Manual-stop' not in pivot_table.columns:
    pivot_table['Manual-stop'] = 0

auto = pivot_table['Auto-stop'].astype(int)
manual = pivot_table['Manual-stop'].astype(int)
total = auto + manual
prop = (auto / total.replace(0, np.nan) * 100).fillna(0).round(2)

# 2) 표용 DataFrame 생성 (+ Total)
df_table = pd.DataFrame({
    "GPU Alloc.": actual_order,
    "Auto-Stop": auto.values,
    "Manual-Stop": manual.values,
    "Proportion (%)": prop.values,
})

df_total = pd.DataFrame([{
    "GPU Alloc.": "Total",
    "Auto-Stop": int(auto.sum()),
    "Manual-Stop": int(manual.sum()),
    "Proportion (%)": round((auto.sum() / (auto.sum() + manual.sum()) * 100) if (auto.sum() + manual.sum()) else 0, 2)
}])

df_table = pd.concat([df_table, df_total], ignore_index=True)

# (선택) 화면에서 먼저 확인
df_table


In [ ]:
_mf_capture_cell(26)


# FIGURE 5
## 0309_execution_time_by_gpu_count_and_termination_reason


In [ ]:
_MF_CELL=28
df_viz = df_1[df_1['status_info(group)'] != 'Error'].copy()
df_viz['used_time_clip'] = df_viz['used_time(minute)'].clip(lower=1.0)

order = ['0 GPU', '0.x GPU', '1 GPU', '2 GPU', '3 GPU', '4 GPU', '1.x–3.x GPU']  # 
labels = [x.replace(' GPU', '') for x in order]

plt.figure(figsize=(15, 7))

ax = sns.violinplot(
    data=df_viz,
    x='gpu_allocated(group)',
    y='used_time_clip',
    hue='status_info(group)',
    order=order,
    hue_order=['Auto-stop', 'Manual-stop'],
    log_scale=(False, True),
    inner='box',
    split=False,
    # palette={'Auto-stop': '#2980B9', 'Manual-stop': '#E67E22'},  # ✅ 키 맞춤
    cut=2
)

ax.set_xlabel("GPU Count Per Container", fontsize=25)
ax.set_ylabel("Run Time (minute)", fontsize=25)
ax.set_xticklabels(labels, fontsize=25)
ax.tick_params(axis='y', labelsize=25)

ax.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
ax.set_ylim(1e-3, 1e7)

ax.legend(loc='upper right', fontsize=22, ncol=2, title=None)
plt.grid(axis='y', which='major', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(28)


# FIGURE 6
## 0313_Hourly distribution of termination reasons for containers by GPU Usage


In [ ]:
_MF_CELL=30
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as mtick

# 1. Figure 및 Subplot 설정
fig, axes = plt.subplots(1, 2, figsize=(28, 10), sharey=False)

titles = ["GPU-Enabled Containers", "Non-GPU Containers"]
datasets = [df_1_gpu, df_1_cpu]

# 2그룹: Auto-stop / Manual-stop
plot_order = ["Auto-stop", "Manual-stop"]
colors = {"Auto-stop": "#1f77b4", "Manual-stop": "#ff7f0e"}
auto_marker_fill = colors["Auto-stop"]

for idx, (ax1, df_subset) in enumerate(zip(axes, datasets)):

    # --- A. 시간대별 그룹 카운트 집계 (Stacked Bar) ---
    hourly_counts = (
        df_subset.groupby(["terminated_time(hour)", "status_info(group)"])
        # df_subset.groupby(["created_time(hour)", "status_info(group)"])
        .size()
        .unstack(fill_value=0)
    )

    # 0~23 시간 모두 보이게(없는 시간은 0)
    hourly_counts = hourly_counts.reindex(range(24), fill_value=0)

    existing_cols = [c for c in plot_order if c in hourly_counts.columns]

    if len(existing_cols) == 0 or hourly_counts[existing_cols].to_numpy().sum() == 0:
        ax1.set_title(titles[idx] + " (NO DATA)", fontsize=35, pad=20)
        ax1.text(
            0.5, 0.5,
            "No plottable counts.\nCheck group labels in 'status_info(group)'.",
            ha="center", va="center",
            transform=ax1.transAxes,
            fontsize=18
        )
        continue

    # stacked bar
    hourly_counts[existing_cols].plot(
        kind="bar",
        stacked=True,
        ax=ax1,
        width=0.8,
        color=[colors[c] for c in existing_cols]
    )

    # X축: 2시간 간격 라벨링
    xticks = range(0, 24, 2)
    ax1.set_xticks(xticks)
    ax1.set_xticklabels(xticks, fontsize=30, rotation=0)

    ax1.set_xlabel("Time of Day", fontsize=30)
    ax1.set_ylim(0, 1600)

    if idx == 0:
        ax1.set_ylabel("Number of Containers", fontsize=30)
        ax1.legend(existing_cols, loc="upper left", fontsize=25)
    else:
        ax1.set_ylabel("")
        if ax1.get_legend():
            ax1.get_legend().remove()

    ax1.tick_params(axis="both", labelsize=30)

    # --- B. 우측 Y축: Auto-Stop Ratio (0~1.0) ---
    ax2 = ax1.twinx()

    total_hourly = hourly_counts[existing_cols].sum(axis=1)
    auto_val = hourly_counts["Auto-stop"] if "Auto-stop" in hourly_counts.columns else 0

    # ✅ 0~1 비율로 계산
    auto_ratio = np.where(
        total_hourly > 0,
        (auto_val / total_hourly),
        np.nan
    )

    ax2.plot(
        hourly_counts.index,
        auto_ratio,
        color="black",
        linestyle="--",
        linewidth=3,
        marker="o",
        markerfacecolor=auto_marker_fill,
        markeredgecolor="black",
        markeredgewidth=1.5,
        markersize=12,
        label="Auto-Stop Ratio"
    )

    ax2.set_ylabel("Auto-Stop Ratio", fontsize=30, labelpad=15)
    ax2.set_ylim(0, 1.0)

    # ✅ 표시를 %로 하고 싶으면(0~1 축 + 0~100% 라벨) 이 줄 유지
    # ax2.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))

    # ✅ 만약 0.0~1.0 숫자 그대로 보이게 하려면 위 줄 대신 아래 사용
    # ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f'))

    ax2.tick_params(axis="y", labelsize=30)

    if idx == 0:
        ax2.legend(loc="upper right", fontsize=25)

    ax1.set_title(titles[idx], fontsize=35, pad=20)

plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(30)


In [ ]:
_MF_CELL=31
# (1) count 테이블
hourly_count = (
    df_1_gpu
#     .groupby('created_time(hour)')['status_info(group)']
        .groupby('terminated_time(hour)')['status_info(group)']
    .value_counts(normalize=False)
    .unstack(fill_value=0)
)

# (2) % 테이블
hourly_pct = (
    round(df_1_gpu
    .groupby('terminated_time(hour)')['status_info(group)']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    * 100, 2)
)

# (3) 합치기: 컬럼명에 suffix 붙여서 한 번에 보기
hourly_summary = hourly_count.add_suffix('_count').join(hourly_pct.add_suffix('_pct'))

hourly_summary


In [ ]:
_mf_capture_cell(31)


In [ ]:
_MF_CELL=32
# (1) count 테이블
hourly_count = (
    df_1
    .groupby('gpu_allocated(group)')['status_info(group)']
    .value_counts(normalize=False)
    .unstack(fill_value=0)
)

# (2) % 테이블 (Total 행 계산 후 다시 계산하기 위해 unstack까지만 진행)
# %는 단순히 합산하면 안 되므로, count 테이블의 Total을 기준으로 나중에 계산합니다.

# (3) 합치기 전 Count 테이블에 Total 행 추가
# 모든 열의 합계를 구한 'Total' 행을 생성합니다.
hourly_count.loc['Total'] = hourly_count.sum(numeric_only=True, axis=0)

# (4) 추가된 Total을 포함하여 % 테이블 다시 생성
# 전체 합계 대비 각 항목의 비중을 계산합니다.
hourly_pct = (hourly_count.div(hourly_count.sum(axis=1), axis=0) * 100).round(2)

# (5) 합치기: 컬럼명에 suffix 붙여서 한 번에 보기
hourly_summary = hourly_count.add_suffix('_count').join(hourly_pct.add_suffix('_pct'))

hourly_summary


In [ ]:
_mf_capture_cell(32)


In [ ]:
_MF_CELL=33



In [ ]:
_mf_capture_cell(33)


In [ ]:
(_MF_OUTPUT/'execution_scope.json').write_text(_mf_json.dumps({
    'mode': _MF_MODE,
    'working_notebook': '04_Analysis/section3/Run01_Section3.ipynb',
    'source_notebook': '04_Analysis/section3/Run01_Section3.ipynb',
    'excluded_original_code_cells': [],
    'status': 'EXECUTED_SCOPE_COMPLETE',
    'table2_population': len(df_selected),
    'other_section3_population': len(df_clean),
    'excluded_above_one_noninteger': excluded_n,
    'fractional_below_one_retained': int(df_clean['gpu_allocated(EA)'].between(0,1,inclusive='neither').sum()),
}, indent=2, ensure_ascii=False))
